In [5]:
import pandas as pd
import scipy.stats as stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Ingests data and drops rows with missing targets or categories
vg_data = pd.read_csv("vgsales.csv").dropna(subset=["Global_Sales", "Platform", "Publisher"])

# Applies One-Hot Encoding instead of categorical codes
# Prevents the tree algorithm from assuming ordinal relationships between categorical variables
vg_encoded = pd.get_dummies(vg_data, columns=["Platform", "Publisher"], drop_first=True)

# Prevents data leakage
# Drops regional sales to ensure the model does not merely memorise NA + EU + JP + Other = Global
leakage_columns = ["Rank", "Name", "Year", "Genre", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
feature_cols = [col for col in vg_encoded.columns if col not in leakage_columns]

X_full = vg_encoded[feature_cols]
y_full = vg_encoded["Global_Sales"]

# Sets up the grid search parameters for exact percentile boundaries
lower_bound_candidates = range(0, 6)   # Evaluates lower tail cuts between 0% and 5%
upper_bound_candidates = range(95, 101)  # Evaluates upper tail cuts between 95% and 100%

best_score = -float('inf')
best_range = (None, None)
best_scores_fold = None

print("--- Executing Automated Grid Search for Exact Optimal Range ---")
for p_low in lower_bound_candidates:
    for p_high in upper_bound_candidates:
        if p_low >= p_high:
            continue
            
        low_val = y_full.quantile(p_low / 100.0)
        high_val = y_full.quantile(p_high / 100.0)
        
        # Filters target and features based on percentiles
        mask = (y_full >= low_val) & (y_full <= high_val)
        X_sub, y_sub = X_full[mask], y_full[mask]
        
        # Performs cross-validation to gather R² score distributions across 5 folds
        # Utilises all CPU cores to handle the wider One-Hot Encoded dataset
        model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
        scores = cross_val_score(model, X_sub, y_sub, cv=5, scoring="r2")
        mean_score = scores.mean()
        
        # Updates the best configuration if the current mean R² outperforms the previous best
        if mean_score > best_score:
            best_score = mean_score
            best_range = (p_low, p_high)
            best_scores_fold = scores

print(f"\nExact Optimal Range Discovered: {best_range[0]}th to {best_range[1]}th Percentile")
print(f"Maximum Cross-Validated Mean R²: {best_score:.4f}")

# Establishes the baseline model using the unfiltered dataset (0-100th percentile)
baseline_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
baseline_scores = cross_val_score(
    baseline_model, 
    X_full, 
    y_full, 
    cv=5, 
    scoring="r2"
)

# Conducts a Paired T-Test to compare the optimised model against the baseline
alpha = 0.05
t_stat, p_val = stats.ttest_rel(best_scores_fold, baseline_scores)

print(f"\n--- Statistical Validation: Optimised {best_range} vs. Raw Baseline ---")
print(f"T-Statistic Result: {t_stat:.4f} | Computed P-Value: {p_val:.4e}")

if p_val < alpha and t_stat > 0:
    print(f"Conclusion: The optimised boundary {best_range} significantly outperforms the raw dataset given an alpha threshold of {alpha}.")

else:
    print("Conclusion: The optimised range failed to show a statistically significant improvement over the baseline.")

--- Executing Automated Grid Search for Exact Optimal Range ---

Exact Optimal Range Discovered: 4th to 95th Percentile
Maximum Cross-Validated Mean R²: -171.5816

--- Statistical Validation: Optimised (4, 95) vs. Raw Baseline ---
T-Statistic Result: 1.8237 | Computed P-Value: 1.4226e-01
Conclusion: The optimised range failed to show a statistically significant improvement over the baseline.
